# Exam Evaluation — AI Text Detection (600 texts)

Models: **model_A** TF-IDF+LR · **model_B** E5+stylo+CatBoost · **model_C** E5+stylo+Kogan+CatBoost · **model_D** DeBERTa-v3-large  
Metrics: ROC-AUC · Brier · C@1 · F1 · F0.5u · MEAN · FPR · FNR

In [ ]:
!pip install -q sentence-transformers catboost textstat spacy
!python -m spacy download en_core_web_sm -q

In [ ]:
import json, re, string, pickle, math
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spacy
import textstat
import torch
from catboost import CatBoostClassifier, Pool
from scipy.sparse import hstack
from scipy.spatial.distance import cdist
from sklearn.cluster import MiniBatchKMeans
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, brier_score_loss, accuracy_score,
    f1_score, fbeta_score, confusion_matrix, roc_curve,
)
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer

try:
    from torch.amp import autocast
except ImportError:
    from torch.cuda.amp import autocast

DEV = "cuda" if torch.cuda.is_available() else "cpu"
N_GPUS = torch.cuda.device_count()
print(f"device: {DEV}  gpus: {N_GPUS}")

NLP = spacy.load("en_core_web_sm", disable=["ner", "parser"])
NLP.add_pipe("sentencizer")


In [ ]:
TRN_FILE = "/kaggle/input/datasets/arinazamyshevskaya/pan25-generative-ai-detection-task1-train/pan25-generative-ai-detection-task1-train/train.jsonl"
EXAM_CSV = "/kaggle/input/datasets/arinazamyshevskaya/test-data-nlp2/dataset_600.csv"
MODEL_B_CBM = "/kaggle/input/datasets/arinazamyshevskaya/models-for-test/model_b.cbm"
MODEL_C_CBM = "/kaggle/input/datasets/arinazamyshevskaya/models-for-test/model_c.cbm"
DEBERTA_PT = "/kaggle/input/datasets/arinazamyshevskaya/models-for-test/deberta_best.pt"
DEBERTA_NAME = "microsoft/deberta-v3-large"
EMB_MODEL = "intfloat/e5-small-v2"
N_CLUSTERS = 20
CHUNK_SENTS = 3
MAX_LEN = 256
INFER_BATCH = 8
OUT_DIR = Path("/kaggle/working")

# thresholds chosen by best_threshold(y_val, proba) in training notebooks
VAL_THRS = {
    "model_A (tfidf+lr)": 0.53,
    "model_B (e5+stylo+catboost)": 0.69,
    "model_C (e5+stylo+kogan)": 0.59,
    "model_D (deberta)": 0.5,
}


## 1. Data

In [ ]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

trn_recs = load_jsonl(TRN_FILE)
y_trn = np.array([r["label"] for r in trn_recs])
print(f"train: {len(trn_recs)}  ai={y_trn.sum()}  human={(y_trn==0).sum()}")

exam_df = pd.read_csv(EXAM_CSV)
exam_df["genre"] = exam_df["genre"].fillna("unknown").astype(str)
print(f"\nexam: {exam_df.shape}")
print(exam_df["condition"].value_counts().to_string())

assert exam_df.shape[0] == 600
assert set(exam_df["condition"].unique()) == {"human", "ai_clean", "ai_obfuscated"}
print("sanity check passed")

texts_exam = exam_df["text"].tolist()
y_exam = exam_df["label"].values


## 2. Features

### 2a. Stylometry

In [ ]:
_STOP = set(ENGLISH_STOP_WORDS)


def tokenize_words(txt):
    return re.findall(r"\b\w+\b", txt.lower())

def tokenize_words_raw(txt):
    return re.findall(r"\b\w+\b", txt)

def tokenize_sents(txt):
    sents = re.split(r"(?<=[.!?])\s+", txt.strip())
    return [s for s in sents if s.strip()]

def _mttr(words, win=100):
    if len(words) < win:
        return len(set(words)) / len(words) if words else 0.0
    return float(np.mean([len(set(words[i:i+win])) / win
                          for i in range(len(words)-win+1)]))

def _hapax_rate(w):
    if not w: return 0.0
    c = Counter(w); return sum(1 for v in c.values() if v == 1) / len(w)

def _dis_legomena_rate(w):
    if not w: return 0.0
    c = Counter(w); return sum(1 for v in c.values() if v == 2) / len(w)

def _yule_k(words):
    if not words: return 0.0
    c = Counter(words); N = len(words)
    M2 = sum(v * f * f for v, f in Counter(c.values()).items())
    return 10000 * (M2 - N) / (N * N)

def _simpson_d(words):
    if not words: return 0.0
    c = Counter(words); N = len(words)
    return sum(v*(v-1) for v in c.values()) / (N*(N-1)) if N > 1 else 0.0

def _burstiness(sents):
    if len(sents) < 2: return 0.0
    lens = [len(tokenize_words(s)) for s in sents]
    mu = np.mean(lens)
    return float(np.std(lens) / mu) if mu else 0.0

def _bigram_uniq(w):
    if len(w) < 2: return 0.0
    bg = list(zip(w, w[1:])); return len(set(bg)) / len(bg)

def _trigram_uniq(w):
    if len(w) < 3: return 0.0
    tg = list(zip(w, w[1:], w[2:])); return len(set(tg)) / len(tg)

def stylometric_ftrs(txt):
    words = tokenize_words(txt); words_raw = tokenize_words_raw(txt)
    sents = tokenize_sents(txt)
    nw = max(len(words), 1); ns = max(len(sents), 1); nc = max(len(txt), 1)
    sl = [len(tokenize_words(s)) for s in sents]
    return {
        "n_chars": nc, "n_words": nw, "n_sents": ns,
        "avg_word_len": float(np.mean([len(w) for w in words])) if words else 0.0,
        "avg_sent_len": nw/ns, "sent_len_std": float(np.std(sl)) if sl else 0.0,
        "sent_len_max": float(max(sl)) if sl else 0.0,
        "sent_len_min": float(min(sl)) if sl else 0.0,
        "ttr": len(set(words))/nw, "mttr": _mttr(words),
        "hapax_rate": _hapax_rate(words),
        "dis_legomena_rate": _dis_legomena_rate(words),
        "yule_k": _yule_k(words), "simpson_d": _simpson_d(words),
        "burstiness": _burstiness(sents),
        "bigram_uniq": _bigram_uniq(words), "trigram_uniq": _trigram_uniq(words),
        "repeated_word_ratio": sum(1 for v in Counter(words).values() if v > 1) / nw,
        "stopword_ratio": sum(1 for w in words if w in _STOP) / nw,
        "punct_dens": sum(1 for c in txt if c in string.punctuation) / nc,
        "comma_ratio": txt.count(",") / ns, "period_ratio": txt.count(".") / ns,
        "question_ratio": txt.count("?") / ns, "exclaim_ratio": txt.count("!") / ns,
        "uppercase_ratio": sum(1 for c in txt if c.isupper()) / nc,
        "digit_ratio": sum(1 for c in txt if c.isdigit()) / nc,
        "lowercase_ratio": sum(1 for w in words_raw if w.islower()) / max(len(words_raw), 1),
    }

def fingerprint_ftrs(txt):
    words = tokenize_words(txt); sents = tokenize_sents(txt)
    ns = max(len(sents), 1)
    paras = [p.strip() for p in txt.split("\n\n") if p.strip()]
    if len(paras) >= 2:
        pl = [len(tokenize_words(p)) for p in paras]; mu = float(np.mean(pl))
        para_burst = float(np.std(pl) / mu) if mu else 0.0
    else:
        para_burst = 0.0
    _SFX = ("ing","tion","ness","ment","er","ly","ies","ed","es","s")
    def _stem(w):
        for s in _SFX:
            if w.endswith(s) and len(w) > len(s)+3: return w[:-len(s)]
        return w
    stems = [_stem(w) for w in words]
    return {
        "think_block": float(bool(re.search(r"<think>", txt, re.IGNORECASE))),
        "boxed_ans": float(r"\boxed{" in txt),
        "em_dash_rate": txt.count("—") / ns,
        "md_density": len(re.findall(r"#{1,6}\s|\*\*|`|\n[-*]\s", txt)) / ns,
        "para_burst": para_burst,
        "stem_voc_rat": len(set(stems)) / len(stems) if stems else 0.0,
    }

_POS_TAGS = ["NOUN","VERB","ADJ","ADV","PRON","PROPN","DET","ADP","CCONJ","SCONJ","NUM"]

def pos_ftrs(txt):
    doc = NLP(txt[:5000])
    tokens = [t for t in doc if not t.is_space]
    n = max(len(tokens), 1)
    counts = Counter(t.pos_ for t in tokens)
    ratios = {f"pos_{p.lower()}": counts.get(p, 0) / n for p in _POS_TAGS}
    total = sum(counts.values())
    probs = [c/total for c in counts.values()] if total else []
    ratios["pos_entropy"] = float(-sum(p*math.log(p+1e-9) for p in probs))
    bigs = list(zip([t.pos_ for t in tokens], [t.pos_ for t in tokens][1:]))
    ratios["pos_bigram_div"] = len(set(bigs)) / max(len(bigs), 1)
    return ratios

def readability_ftrs(txt):
    return {
        "flesch_ease": textstat.flesch_reading_ease(txt),
        "fk_grade": textstat.flesch_kincaid_grade(txt),
        "gunning_fog": textstat.gunning_fog(txt),
        "smog": textstat.smog_index(txt),
        "ari": textstat.automated_readability_index(txt),
        "coleman_liau": textstat.coleman_liau_index(txt),
    }

def all_ftrs(txt):
    d = stylometric_ftrs(txt)
    d.update(fingerprint_ftrs(txt))
    d.update(pos_ftrs(txt))
    d.update(readability_ftrs(txt))
    return d

print("feature functions ready")


### 2b. E5 Embeddings

In [ ]:
emb_model = SentenceTransformer(EMB_MODEL)

def embed(texts, batch_size=64):
    return emb_model.encode(
        [f"passage: {t}" for t in texts],
        batch_size=batch_size, normalize_embeddings=True, show_progress_bar=True,
    )

print("embedding PAN train (needed for Kogan KMeans)...")
trn_texts = [r["text"] for r in trn_recs]
trn_emb = embed(trn_texts)
print(f"train embeddings: {trn_emb.shape}")

print("embedding exam set...")
exam_emb = embed(texts_exam)
print(f"exam  embeddings: {exam_emb.shape}")


### 2c. Kogan Cluster Features

In [ ]:
kmeans = MiniBatchKMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=5)
kmeans.fit(trn_emb)

ai_mask = y_trn == 1
human_c = trn_emb[~ai_mask].mean(axis=0, keepdims=True)
ai_c = trn_emb[ai_mask].mean(axis=0, keepdims=True)


def _doc_ftrs(vec, kmeans, human_c, ai_c):
    v = vec.reshape(1, -1)
    dists = cdist(v, kmeans.cluster_centers_, metric="euclidean")[0]
    nearest = int(np.argmin(dists))
    sorted_d = np.sort(dists)
    return {
        "dist_human": float(cdist(v, human_c)[0, 0]),
        "dist_ai": float(cdist(v, ai_c)[0, 0]),
        "rel_dist": float(cdist(v, human_c)[0, 0] - cdist(v, ai_c)[0, 0]),
        "nearest_cluster": f"c{nearest}",
        "dist_nearest": float(sorted_d[0]),
        "dist_second": float(sorted_d[1]) if len(sorted_d) > 1 else 0.0,
        "cluster_margin": float(sorted_d[1] - sorted_d[0]) if len(sorted_d) > 1 else 0.0,
    }


def _chunk_entropy(texts, emb_model, kmeans, chunk_sents=CHUNK_SENTS, batch_size=256):
    all_chunks, offsets = [], []
    for txt in texts:
        sents = re.split(r"(?<=[.!?])\s+", txt.strip())
        chunks = [" ".join(sents[i:i+chunk_sents])
                  for i in range(0, len(sents), chunk_sents) if sents[i:i+chunk_sents]]
        offsets.append((len(all_chunks), len(all_chunks) + len(chunks)))
        all_chunks.extend(chunks)
    if not all_chunks:
        return [0.0] * len(texts)
    cemb = emb_model.encode(
        [f"passage: {c}" for c in all_chunks],
        batch_size=batch_size, normalize_embeddings=True, show_progress_bar=False,
    )
    result = []
    for s, e in offsets:
        if s == e:
            result.append(0.0); continue
        labels = kmeans.predict(cemb[s:e])
        cnt = np.bincount(labels, minlength=N_CLUSTERS).astype(float)
        cnt /= max(cnt.sum(), 1)
        result.append(float(-np.sum(cnt * np.log(cnt + 1e-9))))
    return result


print("computing Kogan features for exam set...")
exam_kogan = pd.DataFrame([_doc_ftrs(v, kmeans, human_c, ai_c) for v in tqdm(exam_emb)])
exam_kogan["chunk_entropy"] = _chunk_entropy(texts_exam, emb_model, kmeans)
print(f"Kogan features: {exam_kogan.shape}")


### 2d. Stylometry + POS + Readability

In [ ]:
exam_stylo = pd.DataFrame([all_ftrs(t) for t in tqdm(texts_exam)])
print(f"stylo features: {exam_stylo.shape}  columns: {list(exam_stylo.columns)[:6]}...")


## 3. Model A — TF-IDF + Logistic Regression

In [ ]:
trn_texts_plain = [r["text"] for r in trn_recs]

word_vec = TfidfVectorizer(
    analyzer="word", ngram_range=(1, 2), min_df=2,
    max_features=100000, sublinear_tf=True,
)
char_vec = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 5), min_df=2,
    max_features=100000, sublinear_tf=True,
)

X_trn_w = word_vec.fit_transform(trn_texts_plain)
X_trn_c = char_vec.fit_transform(trn_texts_plain)
X_trn_a = hstack([X_trn_w, X_trn_c])

clf_a = LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced",
                            solver="saga", n_jobs=-1)
clf_a.fit(X_trn_a, y_trn)

X_exam_a = hstack([word_vec.transform(texts_exam), char_vec.transform(texts_exam)])
proba_a = clf_a.predict_proba(X_exam_a)[:, 1]
print(f"Model A done  mean_score={proba_a.mean():.3f}")


## 4. Model B — E5 + Stylometry + POS + Readability + CatBoost

In [ ]:
emb_cols = [f"e{i:03d}" for i in range(exam_emb.shape[1])]

exam_b = pd.concat([
    pd.DataFrame(exam_emb, columns=emb_cols),
    exam_stylo.reset_index(drop=True),
], axis=1)
exam_b["genre"] = exam_df["genre"].values
print(f"model_B feature matrix: {exam_b.shape}")

model_b = CatBoostClassifier()
model_b.load_model(MODEL_B_CBM)
pool_b = Pool(exam_b, cat_features=["genre"], feature_names=list(exam_b.columns))
proba_b = model_b.predict_proba(pool_b)[:, 1]
print(f"Model B done  mean_score={proba_b.mean():.3f}")


## 5. Model C — E5 + Stylometry + Kogan + CatBoost

In [ ]:
exam_c = pd.concat([
    exam_b.reset_index(drop=True),
    exam_kogan.reset_index(drop=True),
], axis=1)
print(f"model_C feature matrix: {exam_c.shape}")

model_c = CatBoostClassifier()
model_c.load_model(MODEL_C_CBM)
pool_c = Pool(exam_c, cat_features=["genre", "nearest_cluster"],
              feature_names=list(exam_c.columns))
proba_c = model_c.predict_proba(pool_c)[:, 1]
print(f"Model C done  mean_score={proba_c.mean():.3f}")


## 6. Model D — DeBERTa-v3-Large

In [ ]:
tok_d = AutoTokenizer.from_pretrained(DEBERTA_NAME)
cfg_d = AutoConfig.from_pretrained(DEBERTA_NAME, num_labels=1)
model_d = AutoModelForSequenceClassification.from_config(cfg_d)
state_d = torch.load(DEBERTA_PT, map_location="cpu")
model_d.load_state_dict(state_d, strict=False)
model_d.eval().to(DEV)

proba_d = []
for i in tqdm(range(0, len(texts_exam), INFER_BATCH)):
    batch = texts_exam[i:i+INFER_BATCH]
    enc = tok_d(batch, max_length=MAX_LEN, truncation=True,
                padding=True, return_tensors="pt")
    enc = {k: v.to(DEV) for k, v in enc.items()}
    with torch.no_grad():
        if DEV == "cuda":
            with autocast("cuda"):
                logits = model_d(**enc).logits.squeeze(-1)
        else:
            logits = model_d(**enc).logits.squeeze(-1)
    proba_d.extend(torch.sigmoid(logits.float()).cpu().numpy().tolist())

proba_d = np.array(proba_d)
print(f"Model D done  mean_score={proba_d.mean():.3f}")
del model_d; torch.cuda.empty_cache()


## 7. Evaluation

In [ ]:
def pan_metrics(y_true, proba, thr=0.5):
    y_pred = (proba >= thr).astype(int)
    roc_auc = roc_auc_score(y_true, proba)
    brier = 1.0 - brier_score_loss(y_true, proba)
    c1 = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    f05u = fbeta_score(y_true, y_pred, beta=0.5, pos_label=1, zero_division=0)
    mean = float(np.mean([roc_auc, brier, c1, f1, f05u]))
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    return {
        "ROC-AUC": round(roc_auc, 4), "BRIER": round(brier, 4),
        "C@1": round(c1, 4), "F1": round(f1, 4), "F0.5u": round(f05u, 4),
        "MEAN": round(mean, 4), "FPR": round(fpr, 4), "FNR": round(fnr, 4),
    }

def print_table(rows, title=""):
    if not rows: return
    if title: print(f"\n{title}")
    cols = list(rows[0].keys())
    widths = {c: max(len(c), max(len(str(r.get(c, ""))) for r in rows)) for c in cols}
    hdr = "  ".join(c.ljust(widths[c]) for c in cols)
    print(hdr); print("-" * len(hdr))
    for row in rows:
        print("  ".join(str(row.get(c, "")).ljust(widths[c]) for c in cols))

all_models = {
    "model_A (tfidf+lr)": proba_a,
    "model_B (e5+stylo+catboost)": proba_b,
    "model_C (e5+stylo+kogan)": proba_c,
    "model_D (deberta)": proba_d,
}


### Overall results

In [ ]:
overall_rows = []
for name, proba in all_models.items():
    thr = VAL_THRS[name]
    m = pan_metrics(y_exam, proba, thr=thr)
    overall_rows.append({"system": name, "thr": thr, **m})

print_table(overall_rows, title="OVERALL (all 600)")
pd.DataFrame(overall_rows).to_csv(OUT_DIR / "exam_overall.csv", index=False)


### Per condition

In [ ]:
cond_rows = []
for name, proba in all_models.items():
    rows = []
    for cond_a, cond_b, label in [
        ("human", "ai_plain",      "human vs ai_plain"),
        ("human", "ai_obfuscated", "human vs ai_obfuscated"),
    ]:
        mask = exam_df["condition"].isin([cond_a, cond_b]).values
        sub_y = y_exam[mask]
        if sub_y.sum() == 0 or (sub_y == 0).sum() == 0:
            continue
        m = pan_metrics(sub_y, proba[mask], thr=VAL_THRS[name])
        row = {"condition": label, "n": int(mask.sum()), **m}
        rows.append(row)
        cond_rows.append({"model": name, **row})
    print_table(rows, title=name)

pd.DataFrame(cond_rows).to_csv(OUT_DIR / "exam_by_condition.csv", index=False)


### Per genre

In [ ]:
genre_rows = []
for name, proba in all_models.items():
    rows = []
    for genre in sorted(exam_df["genre"].unique()):
        mask = (exam_df["genre"] == genre).values
        if mask.sum() < 10:
            continue
        sub_y = y_exam[mask]
        if sub_y.sum() == 0 or (sub_y == 0).sum() == 0:
            continue
        m = pan_metrics(sub_y, proba[mask], thr=VAL_THRS[name])
        row = {"genre": genre, "n": int(mask.sum()), **m}
        rows.append(row)
        genre_rows.append({"model": name, **row})
    print_table(rows, title=name)

pd.DataFrame(genre_rows).to_csv(OUT_DIR / "exam_by_genre.csv", index=False)


## 8. Plots

In [ ]:
COLORS = {name: c for name, c in zip(all_models.keys(),
          ["#4e79a7", "#f28e2b", "#59a14f", "#e15759"])}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
for ax, (name, proba) in zip(axes, all_models.items()):
    ax.hist(proba[y_exam == 0], bins=40, alpha=0.6, label="human", color="steelblue")
    ax.hist(proba[y_exam == 1], bins=40, alpha=0.6, label="AI",    color="tomato")
    ax.axvline(VAL_THRS[name], color="black", linestyle="--", linewidth=1.2,
               label=f"thr={VAL_THRS[name]}")
    ax.set_title(name); ax.set_xlabel("score"); ax.legend(fontsize=8)
fig.suptitle("Score distributions — human vs AI", fontsize=13)
fig.tight_layout()
fig.savefig(str(OUT_DIR / "plot_score_dist.png"), dpi=120)
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
for name, proba in all_models.items():
    fpr_c, tpr_c, _ = roc_curve(y_exam, proba)
    auc_val = roc_auc_score(y_exam, proba)
    ax.plot(fpr_c, tpr_c, label=f"{name}  AUC={auc_val:.4f}",
            color=COLORS[name], linewidth=2)
ax.plot([0, 1], [0, 1], "k--", linewidth=1)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC curves — exam set (600 texts)")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(str(OUT_DIR / "plot_roc.png"), dpi=120)
plt.show()

metric_names = ["ROC-AUC", "BRIER", "C@1", "F1", "F0.5u", "MEAN"]
x = np.arange(len(metric_names)); width = 0.2
fig, ax = plt.subplots(figsize=(12, 5))
for i, (name, proba) in enumerate(all_models.items()):
    vals = [pan_metrics(y_exam, proba, thr=VAL_THRS[name])[k] for k in metric_names]
    ax.bar(x + i*width, vals, width, label=name, color=COLORS[name], alpha=0.85)
ax.set_xticks(x + width*1.5); ax.set_xticklabels(metric_names)
ax.set_ylim(0.5, 1.02); ax.set_ylabel("score")
ax.set_title("PAN metrics comparison — exam set")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(str(OUT_DIR / "plot_metrics_bar.png"), dpi=120)
plt.show()

conditions = ["ai_plain", "ai_obfuscated"]
x = np.arange(len(conditions)); width = 0.18
fig, ax = plt.subplots(figsize=(8, 5))
for i, (name, proba) in enumerate(all_models.items()):
    fnrs = []
    for cond in conditions:
        mask = (exam_df["condition"] == cond).values
        preds = (proba[mask] >= VAL_THRS[name]).astype(int)
        fn = ((y_exam[mask] == 1) & (preds == 0)).sum()
        fnrs.append(fn / mask.sum() if mask.sum() > 0 else 0.0)
    ax.bar(x + i*width, fnrs, width, label=name, color=COLORS[name], alpha=0.85)
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(["AI plain\n(no obfuscation)", "AI obfuscated"])
ax.set_ylabel("False Negative Rate (missed AI)")
ax.set_title("Obfuscation impact: FNR by condition")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(str(OUT_DIR / "plot_obfuscation_fnr.png"), dpi=120)
plt.show()


## 9. Error Analysis

In [ ]:
for name, proba in all_models.items():
    thr = VAL_THRS[name]
    preds = (proba >= thr).astype(int)
    df = exam_df.copy()
    df["score"] = proba; df["pred"] = preds

    fp_df = df[(df["label"] == 0) & (df["pred"] == 1)]
    fn_df = df[(df["label"] == 1) & (df["pred"] == 0)]

    df.to_csv(OUT_DIR / f"preds_{name.split()[0]}_exam.csv", index=False)
    fp_df.to_csv(OUT_DIR / f"{name.split()[0]}_false_positives.csv", index=False)
    fn_df.to_csv(OUT_DIR / f"{name.split()[0]}_false_negatives.csv", index=False)

    print(f"{name}  thr={thr}:")
    print(f"  false positives (human -> AI):  {len(fp_df)}")
    print(f"  false negatives (AI -> human):  {len(fn_df)}")
    if len(fn_df):
        print("  fn by condition:", fn_df["condition"].value_counts().to_dict())
    print()
